In [1]:
import meshio
import numpy as np
from pathlib import Path

input_dir = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input")

bulk = meshio.read(input_dir / "mesh5.vtu")

tetra = bulk.cells_dict["tetra"]
phys_tetra = bulk.cell_data_dict["gmsh:physical"]["tetra"]

for tag, name in [(201, "well1"), (202, "well2")]:
    selected = tetra[phys_tetra == tag]

    used_points = np.unique(selected.flatten())

    old_to_new = {old: new for new, old in enumerate(used_points)}

    new_points = bulk.points[used_points]

    new_tetra = np.array(
        [[old_to_new[node] for node in elem] for elem in selected],
        dtype=int
    )

    bulk_node_ids = used_points.astype(np.int64)

    submesh = meshio.Mesh(
        points=new_points,
        cells=[("tetra", new_tetra)],
        point_data={"bulk_node_ids": bulk_node_ids},
        cell_data={
            "gmsh:physical": [np.full(len(new_tetra), tag, dtype=int)]
        }
    )

    out = input_dir / f"{name}.vtu"
    meshio.write(out, submesh)

    print(f"Created {out.name}")
    print("  tetra cells:", len(new_tetra))
    print("  points:", len(new_points))

Created well1.vtu
  tetra cells: 194
  points: 91
Created well2.vtu
  tetra cells: 194
  points: 91
